# Kaggle GPU Notebook - Lab #28
Chạy notebook này trên **Kaggle** và chọn Accelerator là **GPU T4 x2** (hoặc GPU T4 x1).
Notebook này cấu hình vLLM serving, Embedding API và sử dụng Cloudflare Tunnel để kết nối về máy local.

In [ ]:
# 1. Cài đặt các thư viện cần thiết và hạ cấp NumPy để tương thích với scipy/sentence-transformers
!pip install transformers==4.46.3 vllm==0.7.3 fastapi uvicorn mlflow sentence-transformers --quiet
!pip install "numpy<2" --force-reinstall --quiet

In [ ]:
# 2. Gỡ cài đặt torchcodec để tránh lỗi liên kết động gây crash khi load sentence-transformers
!pip uninstall -y torchcodec

In [ ]:
# 3. Tải công cụ cloudflared để thiết lập các tunnel miễn phí mà không cần token
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
# 4. Giải phóng cổng 8001 và khởi chạy vLLM Server chạy ngầm (chờ load model khoảng 60s)
!fuser -k 8001/tcp

import subprocess
import time
import threading

def run_vllm():
    subprocess.run([
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4",
        "--port", "8001",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.5"
    ])

thread = threading.Thread(target=run_vllm, daemon=True)
thread.start()

print("Đang khởi động vLLM server (vui lòng đợi 60 giây)...")
time.sleep(60)
print("vLLM server đã được kích hoạt thành công trên port 8001!")

In [ ]:
# 5. Tạo Cloudflare Tunnel trỏ đến vLLM (port 8001) và in ra URL
import subprocess
import time

vllm_tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)
for _ in range(50):
    line = vllm_tunnel.stdout.readline()
    if ".trycloudflare.com" in line:
        url = "https://" + line.strip().split("https://")[1].split()[0]
        print("\n=== THÀNH CÔNG ===")
        print("vLLM URL (Copy vào .env):", url)
        print("==================\n")
        break

In [ ]:
# 6. Giải phóng cổng 8002 và khởi động Embedding API server ở background
!fuser -k 8002/tcp

with open("embed_srv.py", "w") as f:
    f.write('''
from fastapi import FastAPI
from sentence_transformers import SentenceTransformer
import uvicorn

app = FastAPI()
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

@app.post("/embed")
def embed(data: dict):
    texts = data["texts"]
    embeddings = model.encode(texts).tolist()
    return {"embeddings": embeddings}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8002)
''')

import subprocess
print("Đang khởi động Embedding API server...")
embed_proc = subprocess.Popen(["python", "embed_srv.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

time.sleep(5)
print("\n=== LOG KHỞI ĐỘNG EMBEDDING SERVER ===")
for _ in range(10):
    line = embed_proc.stdout.readline()
    print(line.strip())

In [ ]:
# 7. Tạo Cloudflare Tunnel trỏ đến Embedding API (port 8002) và in ra URL
import subprocess
import time

embed_tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8002"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)
for _ in range(50):
    line = embed_tunnel.stdout.readline()
    if ".trycloudflare.com" in line:
        url = "https://" + line.strip().split("https://")[1].split()[0]
        print("\n=== THÀNH CÔNG ===")
        print("Embedding URL (Copy vào .env):", url)
        print("==================\n")
        break

In [ ]:
# 8. Ghi lại tham số chạy mô hình trên MLflow
import mlflow

mlflow.set_tracking_uri("https://dagshub.com/YOUR_USER/lab28.mlflow")  # Thay bằng URL MLflow của bạn nếu có
mlflow.set_experiment("lab28-integration")

try:
    with mlflow.start_run(run_name="vllm-serving-v1"):
        mlflow.log_param("model", "Qwen2.5-7B-Instruct-GPTQ-Int4")
        mlflow.log_param("max_model_len", 4096)
        mlflow.log_metric("gpu_memory_utilization", 0.85)
    print("Ghi nhận MLflow hoàn tất!")
except Exception as e:
    print("Bỏ qua MLflow tracking:", e)

In [ ]:
# 9. Giữ cho session luôn bận để không bị Kaggle quét và tắt các tiến trình con
import time
print("Notebook đang hoạt động ở chế độ Keep-Alive. Đừng tắt tab này!")
try:
    while True:
        time.sleep(10)
except KeyboardInterrupt:
    print("Dừng chế độ Keep-Alive.")